# NOTEBOOK 08 - EVALUACIÓN mAP (V2, V3, V4)

## Propósito
Evaluar y comparar los modelos V2, V3 y V4 sobre un dataset de test puro (hold-out).

## Modelos:
- **V2**: 500 imgs, GT ruidoso, augmentation básico
- **V3**: 600 imgs, GT ruidoso, augmentation completo
- **V4**: ~10k imgs, GT ruidoso, augmentation completo
- **V5** (futuro): ~615 imgs, GT limpio

## PASO 1: Generar Test Set Puro (sin contaminación de train)

In [ ]:
import os
import shutil
import random
from tqdm import tqdm

def generar_test_set_puro(
    source_img_dir, source_shp_dir,
    dirs_to_blacklist, output_dir,
    sample_size=100, seed=999
):
    print("--- GENERANDO DATASET DE TEST PURO (HOLD-OUT) ---")
    
    # 1. Crear lista negra (imágenes ya usadas en cualquier entrenamiento)
    blacklist = set()
    for d in dirs_to_blacklist:
        if not os.path.exists(d):
            print(f"[WARN] No existe: {d}")
            continue
        for root, dirs, files in os.walk(d):
            for f in files:
                if f.endswith('.tif'):
                    blacklist.add(f)
    
    print(f"Imágenes 'quemadas' (usadas en entrenamiento): {len(blacklist)}")
    
    # 2. Escanear fuente original
    all_tifs = [f for f in os.listdir(source_img_dir) if f.endswith('.tif')]
    all_shps = os.listdir(source_shp_dir)
    
    # 3. Filtrar candidatos vírgenes
    candidates = []
    print("Buscando imágenes vírgenes...")
    
    for tif in tqdm(all_tifs):
        if tif in blacklist:
            continue
        base_name = os.path.splitext(tif)[0]
        match = next(
            (s for s in all_shps if s.startswith(base_name) and s.endswith('.shp')),
            None
        )
        if match:
            candidates.append((tif, match))
    
    print(f"Candidatos vírgenes encontrados: {len(candidates)}")
    
    real_sample_size = min(sample_size, len(candidates))
    
    # 4. Seleccionar muestra
    random.seed(seed)
    selected_pairs = random.sample(candidates, real_sample_size)
    
    # 5. Copiar a carpeta Test
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir)
    
    print(f"Copiando {real_sample_size} imágenes a {output_dir}...")
    for tif, shp in tqdm(selected_pairs):
        shutil.copy2(
            os.path.join(source_img_dir, tif),
            os.path.join(output_dir, tif)
        )
        shp_base = os.path.splitext(shp)[0]
        for f in all_shps:
            if f.startswith(shp_base):
                shutil.copy2(
                    os.path.join(source_shp_dir, f),
                    os.path.join(output_dir, f)
                )
    
    print(f"\nTest set listo: {real_sample_size} imágenes en {output_dir}")


# --- EJECUTAR ---
generar_test_set_puro(
    source_img_dir=r"D:\Silos\Base de datos\procesado_sin_nubes\tiles",
    source_shp_dir=r"D:\Silos\Base de datos\procesado_sin_nubes\tiles\indices_INBNV\masks\polygons",
    dirs_to_blacklist=[
        r"D:\Silos\Dataset_V2",      # V2
        r"D:\Silos\Dataset_V3",      # V3
        r"D:\Silos\Dataset_V4_10k",  # V4
        r"D:\Silos\Golden_Dataset_1000",  # Golden (futuro V5)
    ],
    output_dir=r"D:\Silos\Dataset_Test_Final",
    sample_size=200,
    seed=999
)

## PASO 2: Evaluación comparativa V2 vs V3 vs V4

In [ ]:
import os
import torch
import torchvision
import rasterio
import geopandas as gpd
import numpy as np
import pandas as pd
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.models import ResNet50_Weights
from torchvision.ops import box_iou
from tqdm import tqdm

# --- CONFIGURACIÓN ---
MODELS = {
    'V2': r"D:\Silos\modelo_silos_v2_optimizado.pth",
    'V3': r"D:\Silos\modelo_silos_v3_aug.pth",
    'V4': r"D:\Silos\modelo_silos_v4_10k.pth",
    # 'V5': r"D:\Silos\modelo_silos_v5_clean.pth",  # Descomentar cuando exista
}

TEST_DIR = r"D:\Silos\Dataset_Test_Final"
THRESHOLD = 0.5
IOU_THRESHOLD = 0.15


def get_model_architecture(num_classes=2):
    anchor_sizes = ((16,), (32,), (64,), (128,), (256,))
    aspect_ratios = ((0.2, 0.5, 1.0, 2.0, 5.0),) * len(anchor_sizes)
    anchor_generator = AnchorGenerator(sizes=anchor_sizes, aspect_ratios=aspect_ratios)
    
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
        weights=None,
        weights_backbone=ResNet50_Weights.DEFAULT,
        rpn_anchor_generator=anchor_generator
    )
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model


def preprocess_image(img_path):
    with rasterio.open(img_path) as src:
        try:
            img_data = src.read([1, 2, 3])
        except:
            img_data = src.read()[:3, :, :]
    img_data = np.nan_to_num(img_data.astype(np.float32), nan=0.0)
    max_val = np.max(img_data)
    if max_val > 255.0:
        img_data /= 10000.0
    elif max_val > 1.0:
        img_data /= 255.0
    img_data = np.clip(img_data, 0, 1)
    return torch.as_tensor(img_data, dtype=torch.float32)


def get_gt_boxes(img_name, test_dir):
    """Obtiene bounding boxes del Ground Truth shapefile."""
    base_name = os.path.splitext(img_name)[0]
    img_path = os.path.join(test_dir, img_name)
    
    # Buscar shapefile
    files_in_dir = os.listdir(test_dir)
    shp_name = next(
        (f for f in files_in_dir if f.startswith(base_name) and f.endswith('.shp')),
        None
    )
    
    if not shp_name:
        return torch.zeros((0, 4), dtype=torch.float32)
    
    shp_path = os.path.join(test_dir, shp_name)
    
    with rasterio.open(img_path) as src:
        transform = src.transform
        height = src.height
        width = src.width
    
    boxes = []
    try:
        gdf = gpd.read_file(shp_path)
        if not gdf.empty:
            for geom in gdf.geometry:
                if geom is None:
                    continue
                if geom.geom_type == 'Polygon':
                    xs, ys = zip(*geom.exterior.coords)
                    rows, cols = rasterio.transform.rowcol(transform, xs, ys)
                    y_min = max(0, min(rows))
                    y_max = min(height, max(rows))
                    x_min = max(0, min(cols))
                    x_max = min(width, max(cols))
                    if x_max > x_min + 1 and y_max > y_min + 1:
                        boxes.append([x_min, y_min, x_max, y_max])
    except Exception:
        pass
    
    if len(boxes) == 0:
        return torch.zeros((0, 4), dtype=torch.float32)
    return torch.as_tensor(boxes, dtype=torch.float32)


def evaluate_model(model_path, model_name, device):
    """Evalúa un modelo y retorna métricas."""
    print(f"\n{'='*50}")
    print(f"  Evaluando: {model_name}")
    print(f"{'='*50}")
    
    if not os.path.exists(model_path):
        print(f"ERROR: No existe {model_path}")
        return None
    
    model = get_model_architecture()
    try:
        model.load_state_dict(torch.load(model_path, map_location=device))
    except Exception as e:
        print(f"Error cargando modelo: {e}")
        return None
    
    model.to(device)
    model.eval()
    
    val_images = [f for f in os.listdir(TEST_DIR) if f.endswith('.tif')]
    if not val_images:
        print("Error: Carpeta de Test vacía.")
        return None
    
    true_positives = 0
    false_positives = 0
    false_negatives = 0
    total_gt_boxes = 0
    total_pred_boxes = 0
    all_scores = []
    all_matches = []  # Para calcular AP
    
    for img_name in tqdm(val_images, desc=f"Evaluando {model_name}"):
        img_path = os.path.join(TEST_DIR, img_name)
        
        # Ground Truth
        gt_boxes = get_gt_boxes(img_name, TEST_DIR)
        total_gt_boxes += len(gt_boxes)
        
        # Predicción
        img_tensor = preprocess_image(img_path)
        with torch.no_grad():
            prediction = model([img_tensor.to(device)])
        
        pred_boxes = prediction[0]['boxes'].cpu()
        pred_scores = prediction[0]['scores'].cpu().numpy()
        
        # Filtrar por threshold
        mask = pred_scores >= THRESHOLD
        pred_boxes = pred_boxes[mask]
        pred_scores = pred_scores[mask]
        total_pred_boxes += len(pred_boxes)
        
        if len(gt_boxes) == 0 and len(pred_boxes) == 0:
            continue
        
        if len(gt_boxes) == 0:
            false_positives += len(pred_boxes)
            for s in pred_scores:
                all_scores.append(s)
                all_matches.append(0)
            continue
        
        if len(pred_boxes) == 0:
            false_negatives += len(gt_boxes)
            continue
        
        # Calcular IoU
        iou_matrix = box_iou(pred_boxes, gt_boxes)
        
        gt_matched = set()
        
        # Ordenar predicciones por score (descendente)
        sorted_indices = np.argsort(-pred_scores)
        
        for pred_idx in sorted_indices:
            score = pred_scores[pred_idx]
            all_scores.append(score)
            
            best_iou = 0
            best_gt = -1
            for gt_idx in range(len(gt_boxes)):
                if gt_idx in gt_matched:
                    continue
                iou_val = iou_matrix[pred_idx, gt_idx].item()
                if iou_val > best_iou:
                    best_iou = iou_val
                    best_gt = gt_idx
            
            if best_iou >= IOU_THRESHOLD and best_gt >= 0:
                true_positives += 1
                gt_matched.add(best_gt)
                all_matches.append(1)
            else:
                false_positives += 1
                all_matches.append(0)
        
        false_negatives += len(gt_boxes) - len(gt_matched)
    
    # Métricas
    precision = true_positives / max(true_positives + false_positives, 1)
    recall = true_positives / max(true_positives + false_negatives, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-6)
    
    # AP simplificado (11-point interpolation)
    if len(all_scores) > 0:
        sorted_idx = np.argsort(-np.array(all_scores))
        sorted_matches = np.array(all_matches)[sorted_idx]
        cum_tp = np.cumsum(sorted_matches)
        cum_fp = np.cumsum(1 - sorted_matches)
        precisions = cum_tp / (cum_tp + cum_fp)
        recalls = cum_tp / max(total_gt_boxes, 1)
        
        ap = 0.0
        for t in np.arange(0, 1.1, 0.1):
            prec_at_recall = precisions[recalls >= t]
            if len(prec_at_recall) > 0:
                ap += np.max(prec_at_recall)
        ap /= 11.0
    else:
        ap = 0.0
    
    results = {
        'Modelo': model_name,
        'TP': true_positives,
        'FP': false_positives,
        'FN': false_negatives,
        'GT_Total': total_gt_boxes,
        'Pred_Total': total_pred_boxes,
        'Precision': round(precision, 4),
        'Recall': round(recall, 4),
        'F1': round(f1, 4),
        'AP@0.15': round(ap, 4)
    }
    
    print(f"\nResultados {model_name}:")
    for k, v in results.items():
        print(f"  {k}: {v}")
    
    return results


# --- EJECUTAR EVALUACIÓN ---
device = torch.device('cpu')
all_results = []

for name, path in MODELS.items():
    result = evaluate_model(path, name, device)
    if result:
        all_results.append(result)

# Tabla comparativa
if all_results:
    df = pd.DataFrame(all_results)
    print("\n" + "=" * 70)
    print("  TABLA COMPARATIVA")
    print("=" * 70)
    print(df.to_string(index=False))
    
    # Guardar a CSV
    csv_path = r"D:\Silos\resultados_comparativa.csv"
    df.to_csv(csv_path, index=False)
    print(f"\nResultados guardados en: {csv_path}")

## PASO 3: Visualización comparativa

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if all_results:
    df = pd.DataFrame(all_results)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    models = df['Modelo'].values
    x = np.arange(len(models))
    
    # Precision
    axes[0].bar(x, df['Precision'].values, color=['#e74c3c', '#3498db', '#2ecc71', '#f39c12'][:len(models)])
    axes[0].set_title('Precision')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(models)
    axes[0].set_ylim(0, 1)
    for i, v in enumerate(df['Precision'].values):
        axes[0].text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=10)
    
    # Recall
    axes[1].bar(x, df['Recall'].values, color=['#e74c3c', '#3498db', '#2ecc71', '#f39c12'][:len(models)])
    axes[1].set_title('Recall')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(models)
    axes[1].set_ylim(0, 1)
    for i, v in enumerate(df['Recall'].values):
        axes[1].text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=10)
    
    # F1
    axes[2].bar(x, df['F1'].values, color=['#e74c3c', '#3498db', '#2ecc71', '#f39c12'][:len(models)])
    axes[2].set_title('F1-Score')
    axes[2].set_xticks(x)
    axes[2].set_xticklabels(models)
    axes[2].set_ylim(0, 1)
    for i, v in enumerate(df['F1'].values):
        axes[2].text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=10)
    
    plt.suptitle(f'Comparativa de Modelos (IoU threshold={IOU_THRESHOLD})', fontsize=14)
    plt.tight_layout()
    
    # Guardar figura
    fig_path = r"D:\Silos\comparativa_modelos.png"
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"Figura guardada en: {fig_path}")
    plt.show()
else:
    print("No hay resultados para graficar.")